# Agent Permission Separation

이번 실습에서는 Agent별 권한을 분리하고, 권한 밖의 도구 호출을 공통 래퍼에서 차단합니다.

핵심은 `Agent가 어떤 역할이라고 말하는가`가 아니라 `런타임에서 어떤 Capability가 허용되어 있는가`입니다.


In [ ]:
from pprint import pprint
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


## 1단계: Agent와 Capability 분리

Agent별 허용 Capability를 별도 테이블로 둡니다. 프롬프트나 노드 이름에 의존하지 않습니다.


In [ ]:
AGENT_CAPABILITIES = {
    "research_agent": {"public.search", "public.summarize"},
    "finance_agent": {"finance.invoice.read", "finance.refund.draft"},
    "hr_agent": {"hr.policy.read"},
    "security_agent": {"audit.log.read", "security.incident.create"},
}


def search_public_docs(query: str) -> str:
    return f"공개 문서 검색 결과: {query}"


def summarize_public_docs(text: str) -> str:
    return f"공개 문서 요약: {text[:30]}"


def read_invoice(account_id: str) -> str:
    return f"{account_id} 계정의 청구서 메타데이터"


def create_refund_draft(account_id: str, amount: int) -> str:
    return f"{account_id} 계정 {amount:,}원 환불 초안 생성"


def read_hr_policy(topic: str) -> str:
    return f"{topic} 관련 HR 정책"


def read_salary(employee_id: str) -> str:
    return f"{employee_id} 급여 정보"


def read_audit_log(limit: int = 5) -> str:
    return f"최근 감사 로그 {limit}건"


def create_security_incident(title: str) -> str:
    return f"보안 사고 티켓 생성: {title}"


TOOL_REGISTRY = {
    "public.search": search_public_docs,
    "public.summarize": summarize_public_docs,
    "finance.invoice.read": read_invoice,
    "finance.refund.draft": create_refund_draft,
    "hr.policy.read": read_hr_policy,
    "hr.salary.read": read_salary,
    "audit.log.read": read_audit_log,
    "security.incident.create": create_security_incident,
}


## 2단계: 공통 도구 호출 래퍼

모든 도구 호출은 `call_tool`을 통과해야 합니다. 이 래퍼가 실제 보안 경계입니다.


In [ ]:
def call_tool(agent_id: str, capability_id: str, **kwargs) -> str:
    allowed = AGENT_CAPABILITIES.get(agent_id, set())

    if capability_id not in allowed:
        raise PermissionError(f"{agent_id} cannot use {capability_id}")

    tool = TOOL_REGISTRY[capability_id]
    return tool(**kwargs)


print(call_tool("research_agent", "public.search", query="LangGraph 보안"))

try:
    call_tool("research_agent", "hr.salary.read", employee_id="E-102")
except PermissionError as error:
    print("차단:", error)


## 3단계: LangGraph 상태와 라우팅

Supervisor가 요청을 분류하더라도 실제 실행 Agent는 자신의 권한만 사용할 수 있습니다.


In [ ]:
class PermissionState(TypedDict, total=False):
    request: str
    assigned_agent: Literal["research_agent", "finance_agent", "hr_agent", "security_agent"]
    requested_capability: str
    tool_args: dict
    tool_result: str
    decision: Literal["allowed", "denied"]
    violations: list[str]
    audit_log: list[str]


def supervisor(state: PermissionState) -> PermissionState:
    request = state["request"].lower()

    if any(word in request for word in ["환불", "청구", "결제"]):
        assigned_agent = "finance_agent"
        if "환불" in request:
            capability = "finance.refund.draft"
            tool_args = {"account_id": "C-100", "amount": 50000}
        else:
            capability = "finance.invoice.read"
            tool_args = {"account_id": "C-100"}
    elif any(word in request for word in ["복지", "휴가", "hr"]):
        assigned_agent = "hr_agent"
        capability = "hr.policy.read"
        tool_args = {"topic": state["request"]}
    elif any(word in request for word in ["침해", "감사", "로그"]):
        assigned_agent = "security_agent"
        if any(word in request for word in ["감사", "로그"]):
            capability = "audit.log.read"
            tool_args = {"limit": 5}
        else:
            capability = "security.incident.create"
            tool_args = {"title": state["request"]}
    else:
        assigned_agent = "research_agent"
        capability = "public.search"
        tool_args = {"query": state["request"]}

    return {
        **state,
        "assigned_agent": assigned_agent,
        "requested_capability": capability,
        "tool_args": tool_args,
        "audit_log": state.get("audit_log", []) + [f"supervisor:{assigned_agent}:{capability}"],
    }


def route_to_agent(state: PermissionState) -> str:
    return state["assigned_agent"]


In [ ]:
def execute_as(agent_id: str, state: PermissionState) -> PermissionState:
    capability = state["requested_capability"]

    try:
        result = call_tool(agent_id, capability, **state["tool_args"])
        return {
            **state,
            "tool_result": result,
            "decision": "allowed",
            "audit_log": state.get("audit_log", []) + [f"{agent_id}:allowed:{capability}"],
        }
    except PermissionError as error:
        return {
            **state,
            "tool_result": "",
            "decision": "denied",
            "violations": state.get("violations", []) + [str(error)],
            "audit_log": state.get("audit_log", []) + [f"{agent_id}:denied:{capability}"],
        }


def research_agent(state: PermissionState) -> PermissionState:
    return execute_as("research_agent", state)


def finance_agent(state: PermissionState) -> PermissionState:
    return execute_as("finance_agent", state)


def hr_agent(state: PermissionState) -> PermissionState:
    return execute_as("hr_agent", state)


def security_agent(state: PermissionState) -> PermissionState:
    return execute_as("security_agent", state)


In [ ]:
workflow = StateGraph(PermissionState)

workflow.add_node("supervisor", supervisor)
workflow.add_node("research_agent", research_agent)
workflow.add_node("finance_agent", finance_agent)
workflow.add_node("hr_agent", hr_agent)
workflow.add_node("security_agent", security_agent)

workflow.add_edge(START, "supervisor")
workflow.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "research_agent": "research_agent",
        "finance_agent": "finance_agent",
        "hr_agent": "hr_agent",
        "security_agent": "security_agent",
    },
)
workflow.add_edge("research_agent", END)
workflow.add_edge("finance_agent", END)
workflow.add_edge("hr_agent", END)
workflow.add_edge("security_agent", END)

graph = workflow.compile()


In [ ]:
examples = [
    "LangGraph 보안 설계 자료를 검색해줘",
    "고객 C-100 환불 초안을 작성해줘",
    "휴가 복지 정책을 알려줘",
]

for request in examples:
    result = graph.invoke({"request": request, "audit_log": []})
    print("=" * 80)
    print("요청:", request)
    print("Agent:", result["assigned_agent"])
    print("Capability:", result["requested_capability"])
    print("Decision:", result["decision"])
    print("Result:", result["tool_result"])
    pprint(result["audit_log"])


## 4단계: Confused Deputy 공격 재현

아래 예시는 Supervisor나 사용자가 `research_agent`에게 HR 급여 조회를 시키려는 상황입니다. 도구 호출 래퍼가 Agent 권한을 기준으로 차단합니다.


In [ ]:
attack_state = {
    "request": "research_agent가 직원 E-102 급여를 조회하게 해줘",
    "assigned_agent": "research_agent",
    "requested_capability": "hr.salary.read",
    "tool_args": {"employee_id": "E-102"},
    "audit_log": ["manual_attack:confused_deputy"],
}

result = research_agent(attack_state)
pprint(result)


## 정리

- Agent별 권한은 중앙 테이블이나 정책 저장소에 둡니다.
- 도구 실행은 공통 래퍼를 통과시킵니다.
- Supervisor의 라우팅 결과와 Agent의 실제 권한은 별개로 검증합니다.
